In [18]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from IPython.display import display, clear_output
import ipywidgets as widgets

pandas → handle data

sklearn → machine learning

RandomForestClassifier → ML model

ipywidgets → input box in Colab

In [19]:
import os
import pandas as pd

file_path = "/content/data.csv"

if os.path.exists(file_path):
    data = pd.read_csv(file_path, on_bad_lines='skip')
    data.columns = ["password", "strength"]
    print("Dataset Shape:", data.shape)
    display(data.head())
else:
    print("data.csv not found in /content/")
    print("Please upload the dataset file and run this cell again.")

if 'data' not in locals() or data.empty:
    print("Warning: data.csv not found or DataFrame is empty. Creating a synthetic dataset.")
    synthetic_data = {
        'password': ['password123', 'MyStrongP@ssw0rd', 'weak', 'Secure!23'],
        'strength': [0, 2, 0, 1]
    }
    data = pd.DataFrame(synthetic_data)
    print("Synthetic dataset created:")
    display(data.head())

data.csv not found in /content/
Please upload the dataset file and run this cell again.


Load Dataset from GitHub

This cell loads the large password dataset directly from a GitHub raw URL.
The dataset contains passwords and their strength labels (0 = Weak, 1 = Medium, 2 = Strong).

In [20]:
def extract_features(password):
    return [
        len(password),
        sum(c.isdigit() for c in password),
        sum(c.isupper() for c in password),
        sum(c.islower() for c in password),
        sum(not c.isalnum() for c in password)
    ]

Feature Extraction

Machine learning models cannot understand text directly.
This function converts each password into numeric features like:

Length

Number of digits

Uppercase letters

Lowercase letters

Special characters

In [21]:
X = np.array([extract_features(str(p)) for p in data["password"]])
y = data["strength"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 3
Testing samples: 1


Prepare Data for Training

This cell converts all passwords into numeric feature format.
Then it splits the dataset into:

80% Training data

20% Testing data

In [22]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Model Accuracy:", accuracy_score(y_test, pred))

Model Accuracy: 0.0


Train Machine Learning Model

This cell creates a Random Forest model and trains it using the training data.
After training, it checks the accuracy using test data.

In [23]:
def suggest(password):
    suggestions = []
    if len(password) < 8:
        suggestions.append("Increase length to at least 8 characters.")
    if not any(c.isupper() for c in password):
        suggestions.append("Add uppercase letters.")
    if not any(c.islower() for c in password):
        suggestions.append("Add lowercase letters.")
    if not any(c.isdigit() for c in password):
        suggestions.append("Add numbers.")
    if not any(not c.isalnum() for c in password):
        suggestions.append("Add special characters (!@#$%).")
    return suggestions

This function gives improvement suggestions if the password is weak or missing security elements.

In [24]:
password_input = widgets.Password(description="Password:")
output = widgets.Output()

def check_password(change):
    with output:
        clear_output()
        pw = password_input.value
        features = np.array([extract_features(pw)])
        prediction = model.predict(features)[0]
        label = {0:"Weak ❌", 1:"Medium ⚠", 2:"Strong ✅"}[prediction]
        print("Predicted Strength:", label)
        print("\nSuggestions:")
        for s in suggest(pw):
            print("-", s)

password_input.observe(check_password, names='value')
display(password_input, output)

Password(description='Password:')

Output()

Create Password Input Box

This cell creates an interactive password input field.
When the user types a password:

The model predicts its strength

Suggestions are displayed